In [1]:
import torch
import cv2
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array

In [1]:
# Load pretrained YOLOv5 model from Ultralytics (replace with your own model if needed)
from ultralytics import yolo
yolo_model = YOLO('yolov5s.pt')

# Load your CNN classification model trained on 12 classes
cnn_model = load_model('C:/Users/Angelina/Desktop/Waste-Classification/models/densenet121_waste_classifier.keras')

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\Angelina\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


ImportError: cannot import name 'yolo' from 'ultralytics' (C:\Users\Angelina\Desktop\Waste-Classification\venv\Lib\site-packages\ultralytics\__init__.py)

In [ ]:
# Hardcoded class names for your CNN classifier
cnn_class_names = ['class1', 'class2', 'class3', 'class4', 'class5', 'class6', 'class7', 'class8', 'class9', 'class10', 'class11', 'class12']

def preprocess_crop(crop_img):
    """Preprocess the cropped image for CNN classification"""
    crop_img = cv2.resize(crop_img, (224, 224))  # Change size according to your CNN input
    crop_img = crop_img.astype("float") / 255.0
    crop_img = img_to_array(crop_img)
    crop_img = np.expand_dims(crop_img, axis=0)
    return crop_img

def detect_and_classify(image_path):
    image = cv2.imread(image_path)
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Run YOLO detection
    results = yolo_model(rgb_image)

    # Get detections: xyxy coordinates, confidence, and class predictions from YOLO
    detections = results.xyxy[0].numpy()

    for *box, conf, cls in detections:
        xmin, ymin, xmax, ymax = map(int, box)

        # Crop detected object
        crop = image[ymin:ymax, xmin:xmax]

        # Preprocess crop for CNN and predict
        processed_crop = preprocess_crop(crop)
        preds = cnn_model.predict(processed_crop)
        pred_index = np.argmax(preds[0])
        label = cnn_class_names[pred_index]
        confidence = preds[0][pred_index]

        # Draw bounding box and label on original image
        cv2.rectangle(image, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)
        text = f"{label}: {confidence*100:.1f}%"
        cv2.putText(image, text, (xmin, ymin - 10), cv2.FONT_HERSHEY_SIMPLEX,
                    0.5, (0, 255, 0), 2)

    # Display the image with bounding boxes and labels
    cv2.imshow("Detection + Classification", image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

# Usage example
detect_and_classify('path/to/your/image.jpg')
